# 🧠 Build a GPT-style LLM from Scratch (PyTorch only)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/loki52501/My_own_llm/blob/main/llm-from-scratch/LLM_From_Scratch.ipynb)

This notebook builds a small **GPT-style Large Language Model** end-to-end, using **only PyTorch** (plus the Python standard library, and matplotlib for plots).
Nothing is imported from `transformers`, `tiktoken`, `tokenizers`, or any other ML library. Every piece is written by hand.

It trains on **your own books**: the `txt-files.tar` archive of Project Gutenberg books in the Hugging Face dataset [`Lokeshlks/gutenberg_books_text`](https://huggingface.co/datasets/Lokeshlks/gutenberg_books_text). It streams just the part it needs.

### The roadmap

```
 ┌───────────┐   ┌────────────┐   ┌──────────────┐   ┌────────────┐   ┌───────────┐   ┌──────────┐   ┌────────────┐
 │ 1. Books  │──▶│ 2. BPE     │──▶│ 3. Dataset / │──▶│ 4. Embed-  │──▶│ 5. Self-  │──▶│ 6. GPT   │──▶│ 7. Train & │
 │  (.txt)   │   │ Tokenizer  │   │  DataLoader  │   │   dings    │   │ Attention │   │  Model   │   │  Generate  │
 └───────────┘   └────────────┘   └──────────────┘   └────────────┘   └───────────┘   └──────────┘   └────────────┘
   raw text       text → ids       (input, target)    ids → vectors   tokens "talk"   stack blocks    learn + write
```

| Step | What you build | Key idea |
|---|---|---|
| 1 | Load the books | An LLM only ever sees one long stream of text |
| 2 | Byte-Pair-Encoding tokenizer | Repeatedly merge the most frequent pair of symbols |
| 3 | Sliding-window dataset | Target = input shifted one token to the right |
| 4 | Token + positional embeddings | Every token id becomes a learnable vector, plus "where am I" information |
| 5 | Causal self-attention | Each token builds its meaning from a weighted mix of earlier tokens |
| 6 | Transformer block → GPT | Attention + MLP + LayerNorm + residuals, stacked N times |
| 7 | Training loop and sampling | Minimize next-token cross-entropy, then generate text one token at a time |

> **Colab tip:** pick a GPU runtime (A100 or L4 is fastest, T4 works) and add a Hugging Face token as the Colab secret `HF_TOKEN` (see section 0b). The long work (tokenizing **every `.txt` file** of the 30 GB tar, then training on all of it) runs in a **background process** that survives page reloads. Progress is saved to your private Hugging Face repo, so if Colab recycles the runtime, **Run all** continues where it stopped. No Google Drive needed. It also runs on CPU, as a small demo.

This follows the same order as Sebastian Raschka's *Build a Large Language Model (From Scratch)* and Andrej Karpathy's *nanoGPT* / *minbpe*.

## 0. Setup

We only need `torch` (already installed on Colab) and `matplotlib` for plots. This cell imports everything, fixes the random seed, and picks the device.

In [ ]:
import os, re, io, sys, math, time, json, glob, heapq, tarfile, random, urllib.request, collections
from pathlib import Path

import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

IN_COLAB = "google.colab" in sys.modules

SEED = 1337
random.seed(SEED)
torch.manual_seed(SEED)

device = "cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu")
print(f"PyTorch {torch.__version__} | device = {device} | running in Colab = {IN_COLAB}")
if device == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))

### 0b. Where progress is saved: a private Hugging Face repo 🛟

Processing and training on the whole 30 GB tar takes many hours. Colab pages reload, tabs disconnect, and runtimes get recycled. So this notebook is built to survive all of that:

```
 ┌─────────────── Colab runtime ────────────────┐          ┌──── Hugging Face (private repo) ────┐
 │ notebook cells (can reload / crash: fine)     │          │ tokenizer.json                      │
 │                                               │  upload  │ progress.json   (tar byte offset)   │
 │ run.py  ← separate background process         │ ───────▶ │ tokens_00000.bin … (1 GB shards)    │
 │   1) stream tar → tokens (shard by shard)     │          │ train_state.pt  (weights+optimizer) │
 │   2) train on all shards (checkpoints)        │ ◀─────── │ gpt_from_scratch.pt                 │
 │   writes run.log                              │  resume  │                                     │
 └───────────────────────────────────────────────┘          └─────────────────────────────────────┘
```

* The heavy work runs in **`run.py`, a background process**. Closing or reloading the page doesn't stop it.
* Every finished 1 GB token shard, and every checkpoint (every `CKPT_MINUTES`), is **uploaded to your private repo**.
* If the runtime is recycled, just **Run all** again. Everything is downloaded back, and the work **continues where it stopped**.

**One-time setup:**
1. Create a Hugging Face token with **write** access at <https://huggingface.co/settings/tokens>.
2. In Colab, open the 🔑 **Secrets** panel (left sidebar), add a secret named `HF_TOKEN`, paste the token, and enable *Notebook access*.

(Without a token everything still works, but progress is only kept on this runtime.)

In [ ]:
HF_TOKEN = None
try:
    if IN_COLAB:
        from google.colab import userdata
        HF_TOKEN = userdata.get("HF_TOKEN")
    else:
        HF_TOKEN = os.environ.get("HF_TOKEN")
except Exception as e:
    print("no HF_TOKEN secret found:", type(e).__name__)

USE_HF = bool(HF_TOKEN)
if USE_HF:
    from huggingface_hub import HfApi, hf_hub_download
    hf_api = HfApi(token=HF_TOKEN)
    HF_REPO = f"{hf_api.whoami()['name']}/llm-from-scratch-run"      # private model repo for all progress
    hf_api.create_repo(HF_REPO, private=True, exist_ok=True)
    print(f"✅ progress will be saved to https://huggingface.co/{HF_REPO} (private)")
else:
    HF_REPO = None
    print("⚠️  no Hugging Face token: progress stays on this runtime only (lost if it's recycled)")

def hf_fetch(filename, local_dir):
    """Download one file from the progress repo if it exists there. Returns True on success."""
    if not USE_HF:
        return False
    try:
        hf_hub_download(HF_REPO, filename, token=HF_TOKEN, local_dir=local_dir)
        return True
    except Exception:
        return False

### 0c. Reuse a copy of `llm_work` saved on Google Drive (optional)

If you copied the finished work folder to Google Drive (the tokenizer, `progress.json`, the `tokens_*.bin` shards and any checkpoints), point `DRIVE_COPY_DIR` at it. The files are copied back into this runtime's `llm_work/`, so nothing has to be re-tokenized or downloaded. Set `DRIVE_COPY_DIR = None` to skip this (Drive is then never mounted).

In [ ]:
import shutil
DRIVE_COPY_DIR = "/content/drive/MyDrive/llm_work_copy"      # None = don't use Google Drive
_WORK = Path("llm_work"); _WORK.mkdir(exist_ok=True)

if DRIVE_COPY_DIR and IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
if DRIVE_COPY_DIR and Path(DRIVE_COPY_DIR).exists():
    src = Path(DRIVE_COPY_DIR)
    wanted = ["tokenizer.json", "progress.json", "train_state.pt", "gpt_from_scratch.pt"] + \
             sorted(p.name for p in src.glob("tokens_*.bin"))
    t0 = time.time()
    for name in wanted:
        s_, d_ = src / name, _WORK / name
        if not s_.exists():
            continue
        if d_.exists() and d_.stat().st_size == s_.stat().st_size:
            print(f"  ✓ {name} (already here)"); continue
        shutil.copy(s_, d_)
        print(f"  ← {name}  {s_.stat().st_size/1e9:6.2f} GB  ({time.time()-t0:.0f}s)")
    print(f"copied from {src} in {time.time()-t0:.0f}s")
elif DRIVE_COPY_DIR:
    print(f"⚠️  {DRIVE_COPY_DIR} not found. Check the folder name in Google Drive (e.g. MyDrive/llm_work_copy)")

## 1. The books: a 30 GB tar, streamed 📚

The books come from the Hugging Face dataset **[`Lokeshlks/gutenberg_books_text`](https://huggingface.co/datasets/Lokeshlks/gutenberg_books_text)**: one `txt-files.tar` that holds **every** Project Gutenberg `.txt` file, **≈30 GB** in total.

A `.tar` is just files laid end to end, so Python's `tarfile` in **stream mode** (`"r|"`) can read it file by file while it downloads, with no seeking and no need to store the tar.

```
 Hugging Face: txt-files.tar (30 GB)
 ┌────────┬────────┬────────┬────────┬─────── ··· ────────┐
 │ file 1 │ file 2 │ file 3 │ file 4 │  ...  ~100k files   │
 └────────┴────────┴────────┴────────┴─────── ··· ────────┘
      │ HTTP stream (resumable at any byte with a "Range" request)
      ▼
 tarfile "r|" ─▶ one .txt at a time ─▶ strip Gutenberg licence text ─▶ ...
```

* **Every `.txt` file is used**, in every language. (`ENGLISH_ONLY = True` would keep only `Language: English` books.) From each file we only cut the Project Gutenberg header and licence text, which is the same boilerplate in every book.
* `stream_books()` also remembers the **byte position** after each file, and it automatically reconnects from there if the network drops.

In this cell we read only a **small sample from the start** (`SAMPLE_MB`). It's used to train the tokenizer and for the demos. **The whole tar is processed later, in section 3b.**


> **Tip:** the Hugging Face dataset viewer reads this tar as a *WebDataset*: one row per book, with the text in a `txt` column. With the `datasets` library, `load_dataset("Lokeshlks/gutenberg_books_text", split="train", streaming=True)` yields the same rows. Here we stream with the Python standard library, so there's nothing extra to install, and we can resume at an exact byte position.

In [ ]:
import zlib, http.client

HF_TAR_URL    = "https://huggingface.co/datasets/Lokeshlks/gutenberg_books_text/resolve/main/txt-files.tar"
DATASET_BYTES = 30_245_222_400               # size of txt-files.tar
ENGLISH_ONLY  = False                        # False = every .txt file in the tar
SAMPLE_MB     = 50 if torch.cuda.is_available() else 4

WORK_DIR = Path("llm_work"); WORK_DIR.mkdir(exist_ok=True)
EOT = "<|endoftext|>"

def clean(text):
    text = text.replace("\r\n", "\n").replace("\r", "\n").replace("﻿", "")
    # keep only the book itself: drop the Project Gutenberg header and licence footer
    start = re.search(r"\*\*\* ?START OF (?:THE|THIS) PROJECT GUTENBERG.*?\*\*\*", text)
    end = re.search(r"\*\*\* ?END OF (?:THE|THIS) PROJECT GUTENBERG", text)
    if start:
        text = text[start.end():end.start() if end else None]
    return re.sub(r"\n{3,}", "\n\n", text).strip()

def stream_books(start_offset=0, max_retries=50):
    """Yield (name, text, next_offset) for every .txt file in the tar, starting at a byte offset.

    next_offset = where the *following* file starts, i.e. where to resume after this one.
    Network hiccups are handled by re-opening the stream at the last good position."""
    offset, retries = start_offset, 0
    while True:
        base = offset
        try:
            req = urllib.request.Request(HF_TAR_URL, headers={"Range": f"bytes={base}-"} if base else {})
            with urllib.request.urlopen(req, timeout=60) as src, tarfile.open(fileobj=src, mode="r|") as tar:
                for m in tar:
                    nxt = base + m.offset_data + (m.size + 511) // 512 * 512    # tar pads files to 512 bytes
                    if m.isfile() and m.name.lower().endswith(".txt"):
                        raw = tar.extractfile(m).read().decode("utf-8", errors="ignore")
                        if not ENGLISH_ONLY or re.search(r"^Language:\s*English", raw[:5000], re.M):
                            text = clean(raw)
                            if text:                                     # only empty files are skipped
                                yield m.name, text, nxt
                    offset = nxt
            return                                                      # reached the end of the tar
        except (OSError, EOFError, tarfile.TarError, http.client.HTTPException) as e:
            retries += 1
            if retries > max_retries:
                raise
            print(f"  stream interrupted ({e!r}); resuming at byte {offset:,}")
            time.sleep(min(60, 2 * retries))

# ---- a small sample from the start of the tar (tokenizer training + demos) ----
sample_path = WORK_DIR / f"sample_{SAMPLE_MB}MB.json"
if sample_path.exists():
    sample = json.loads(sample_path.read_text())
else:
    sample, n, t0 = {"train": [], "val": []}, 0, time.time()
    for i, (name, text, offset) in enumerate(stream_books(0)):
        sample["val" if i % 50 == 0 else "train"].append(text)       # a few files for quick demos of validation
        n += len(text)
        if n >= SAMPLE_MB * 1e6:
            break
    sample_path.write_text(json.dumps(sample))
    print(f"streamed a {n/1e6:.0f} MB sample in {time.time()-t0:.0f}s")

raw_text = f"\n{EOT}\n".join(sample["train"])
val_text = f"\n{EOT}\n".join(sample["val"])
print(f"sample: {len(sample['train']) + len(sample['val'])} files, {(len(raw_text) + len(val_text))/1e6:.1f} MB "
      f"(the whole {DATASET_BYTES/1e9:.1f} GB tar is processed in section 3b)")

In [ ]:
print("--- sample ---")
print(raw_text[20000:20600])

## 2. The tokenizer ✂️

Neural networks work on numbers, not strings. A **tokenizer** turns text into a list of integers (token ids) and back again.

```
 "The cat sat."  ──encode──▶  [464, 3797, 3332, 13]  ──decode──▶  "The cat sat."
```

### 2a. The simplest tokenizer: characters

Give every unique character an id. It's easy, but sequences get long (1 token per character), which wastes the model's limited context window.

In [ ]:
chars = sorted(set(raw_text))
stoi = {ch: i for i, ch in enumerate(chars)}
itos = {i: ch for ch, i in stoi.items()}

char_encode = lambda s: [stoi[c] for c in s]
char_decode = lambda ids: "".join(itos[i] for i in ids)

s = "Hello, world!"
print(f"char vocab size = {len(chars)}")
print(s, "→", char_encode(s), "→", repr(char_decode(char_encode(s))))

### 2b. Byte-Pair Encoding (BPE), the way GPT does it

BPE starts from **raw bytes** (256 base tokens, so *any* text can be encoded, even emoji) and **learns merges**:

```
 Step 0 (bytes):   t h e _ c a t _ t h e _ h a t
 Most frequent pair: (t,h) → new token 256 "th"
 Step 1:           th e _ c a t _ th e _ h a t
 Most frequent pair: (th,e) → new token 257 "the"
 Step 2:           the _ c a t _ the _ h a t
 Most frequent pair: (a,t) → new token 258 "at"
 Step 3:           the _ c at _ the _ h at
 ...repeat until the vocabulary reaches VOCAB_SIZE
```

Frequent words end up as a single token and rare words get split into pieces, so nothing is ever "out of vocabulary".

**Pre-tokenization.** Like GPT-2, we first split the text into word-like chunks with a regex (`" cat"`, `","`, `"'s"`, ...). Merges never cross chunk boundaries. This keeps tokens sensible, and it makes training fast because we only need to count each *unique* chunk once, weighted by how often it appears.

**Speed trick.** Rather than recount every pair after each merge, we keep a running `pair → count` table and only update the words that contain the merged pair. A heap finds the most frequent pair quickly.

In [ ]:
# GPT-2-style pre-tokenizer, pure `re`:  contractions | words | 1-3 digit numbers | punctuation | whitespace
# ([^\W\d_] = any Unicode letter, so accented words like 'café' stay together)
PRETOKENIZE = re.compile(r"""'(?:[sdmt]|ll|ve|re)| ?[^\W\d_]+| ?\d{1,3}| ?[^\s\w]+| ?_+|\s+(?!\S)|\s+""")

print(PRETOKENIZE.findall("Hello there, I'm learning LLMs in 2024!!  Café au lait."))

In [ ]:
def merge_ids(ids, pair, new_id):
    """Replace every occurrence of `pair` in `ids` with `new_id`."""
    out, i = [], 0
    while i < len(ids):
        if i < len(ids) - 1 and ids[i] == pair[0] and ids[i + 1] == pair[1]:
            out.append(new_id); i += 2
        else:
            out.append(ids[i]); i += 1
    return out


# <|endoftext|> separates books. The other six are reserved for the reasoning notebook
# (Reasoning_Model_From_Scratch.ipynb), which fine-tunes the model trained here.
SPECIAL_TOKENS = [EOT, "<|user|>", "<|assistant|>", "<think>", "</think>", "<answer>", "</answer>"]

class BPETokenizer:
    def __init__(self, special_tokens=SPECIAL_TOKENS):
        self.merges = {}                                   # (id_a, id_b) -> new_id   (order = priority)
        self.vocab = {i: bytes([i]) for i in range(256)}   # id -> bytes
        self.special_tokens = list(special_tokens)
        self.special = {}                                  # "<|endoftext|>" -> id
        self._cache = {}

    # ---------------------------------------------------------------- training
    def train(self, text, vocab_size, verbose_every=0):
        n_merges = vocab_size - 256 - len(self.special_tokens)
        assert n_merges >= 0

        # 1) count unique chunks (special tokens are removed from training text)
        for sp in self.special_tokens:
            text = text.replace(sp, " ")
        word_freq = collections.Counter(PRETOKENIZE.findall(text))
        words = [list(w.encode("utf-8")) for w in word_freq]     # each word as a list of byte ids
        freqs = list(word_freq.values())

        # 2) initial pair counts + index "which words contain this pair"
        pair_counts = collections.defaultdict(int)
        where = collections.defaultdict(set)
        for wi, (ids, f) in enumerate(zip(words, freqs)):
            for p in zip(ids, ids[1:]):
                pair_counts[p] += f
                where[p].add(wi)
        heap = [(-c, p) for p, c in pair_counts.items()]
        heapq.heapify(heap)

        # 3) merge loop
        for i in range(n_merges):
            # pop until the heap top matches the *current* count (lazy deletion)
            while heap:
                neg_c, pair = heapq.heappop(heap)
                if pair_counts.get(pair, 0) == -neg_c and -neg_c > 0:
                    break
            else:
                print("no more pairs to merge"); break

            new_id = 256 + i
            self.merges[pair] = new_id
            self.vocab[new_id] = self.vocab[pair[0]] + self.vocab[pair[1]]
            if verbose_every and (i < 10 or i % verbose_every == 0):
                print(f"merge {i+1:5d}: {self.vocab[pair[0]]!r:>12} + {self.vocab[pair[1]]!r:<12} -> {new_id} {self.vocab[new_id]!r}  (count {-neg_c:,})")

            # update only the words containing this pair
            changed = set()
            for wi in list(where[pair]):
                old = words[wi]
                new = merge_ids(old, pair, new_id)
                if new == old:
                    continue
                f = freqs[wi]
                for p in zip(old, old[1:]):
                    pair_counts[p] -= f
                    changed.add(p)
                for p in zip(new, new[1:]):
                    pair_counts[p] += f
                    where[p].add(wi)
                    changed.add(p)
                words[wi] = new
            for p in changed:
                if pair_counts[p] > 0:
                    heapq.heappush(heap, (-pair_counts[p], p))
            del pair_counts[pair], where[pair]

        # 4) special tokens get the last ids
        for sp in self.special_tokens:
            idx = len(self.vocab)
            self.special[sp] = idx
            self.vocab[idx] = sp.encode("utf-8")
        self._cache = {}
        return self

    @property
    def vocab_size(self):
        return len(self.vocab)

    # ---------------------------------------------------------------- encode / decode
    def _encode_chunk(self, chunk):
        if chunk in self._cache:
            return self._cache[chunk]
        if len(self._cache) > 500_000:          # streaming 30 GB: keep the cache from growing forever
            self._cache.clear()
        ids = list(chunk.encode("utf-8"))
        while len(ids) >= 2:
            # apply the merge that was learned *earliest* (lowest new id) first
            pair = min(zip(ids, ids[1:]), key=lambda p: self.merges.get(p, float("inf")))
            if pair not in self.merges:
                break
            ids = merge_ids(ids, pair, self.merges[pair])
        self._cache[chunk] = ids
        return ids

    def encode(self, text):
        ids = []
        # split out special tokens so they map to their own single id
        if self.special:
            pattern = "(" + "|".join(re.escape(s) for s in self.special) + ")"
            parts = re.split(pattern, text)
        else:
            parts = [text]
        for part in parts:
            if part in self.special:
                ids.append(self.special[part])
            elif part:
                for chunk in PRETOKENIZE.findall(part):
                    ids.extend(self._encode_chunk(chunk))
        return ids

    def decode(self, ids):
        return b"".join(self.vocab[i] for i in ids).decode("utf-8", errors="replace")

    # ---------------------------------------------------------------- save / load
    def save(self, path):
        json.dump({"merges": [[a, b, n] for (a, b), n in self.merges.items()],
                   "special": self.special}, open(path, "w"))

    @classmethod
    def load(cls, path):
        d = json.load(open(path))
        tok = cls(special_tokens=list(d["special"]))
        for a, b, n in d["merges"]:
            tok.merges[(a, b)] = n
            tok.vocab[n] = tok.vocab[a] + tok.vocab[b]
        for sp, i in d["special"].items():
            tok.special[sp] = i
            tok.vocab[i] = sp.encode("utf-8")
        return tok

### Train the tokenizer

`VOCAB_SIZE` is a trade-off. Bigger means fewer tokens per sentence (more text fits in the context), but a bigger embedding table and rarer tokens. GPT-2 uses 50,257. For a ~10M-parameter model, **8,192** is a good fit (4,096 on CPU).

The merges are learned from the **sample** (a few tens of MB is plenty to learn sub-words; training BPE in pure Python on 30 GB would take days). The **same tokenizer is then applied to every file of the tar** by the background job. It's saved to `llm_work/tokenizer.json` and uploaded to your Hugging Face repo. On a new runtime it's **downloaded instead of retrained**, because the token ids must never change mid-project.

In [ ]:
VOCAB_SIZE = 8192 if torch.cuda.is_available() else 4096
TOK_PATH = WORK_DIR / "tokenizer.json"

# ♻️ Already have a tokens.bin from an earlier run that tokenized the whole tar? Reuse it and skip
#    re-tokenizing. Token ids only make sense with the tokenizer that produced them, so also point
#    to the tokenizer.json from that SAME run. (Paths in this runtime; a Drive path works too.)
EXISTING_TOKENS_BIN = "llm_work/tokens.bin"          # None = tokenize the whole tar again
EXISTING_TOKENIZER  = "llm_work/tokenizer.json"      # its tokenizer.json (if lost, it's rebuilt + verified below)

if EXISTING_TOKENS_BIN and str(EXISTING_TOKENS_BIN).startswith("/content/drive") and IN_COLAB:
    from google.colab import drive; drive.mount("/content/drive")
REUSE_TOKENS = bool(EXISTING_TOKENS_BIN) and Path(EXISTING_TOKENS_BIN).exists()

if REUSE_TOKENS and not Path(EXISTING_TOKENIZER).exists():
    # The old tokenizer.json is lost, but BPE training is deterministic: the old run trained on the first
    # 40M characters of this same 50 MB sample with 8,192 tokens. Rebuild it, then PROVE it's identical by
    # re-tokenizing the first files of the tar and comparing with the start of tokens.bin.
    print("old tokenizer.json not found → rebuilding it (same sample, first 40M chars, 8,192 tokens)…")
    t0 = time.time()
    rebuilt = BPETokenizer().train(raw_text[:40_000_000], 8192)
    n_check = min(1_000_000, Path(EXISTING_TOKENS_BIN).stat().st_size // 2)
    old_ids = torch.from_file(str(EXISTING_TOKENS_BIN), shared=False, size=n_check, dtype=torch.int16).long()
    new_ids = []
    for name, text, _ in stream_books(0):
        new_ids += rebuilt.encode(text) + [rebuilt.special[EOT]]
        if len(new_ids) >= n_check:
            break
    same = int((torch.tensor(new_ids[:n_check]) == old_ids).sum())
    rebuilt.save(TOK_PATH)
    print(f"  rebuilt in {time.time()-t0:.0f}s | {same:,} of the first {n_check:,} token ids in tokens.bin match")
    if same == n_check:
        EXISTING_TOKENIZER = str(TOK_PATH)
        print("  ✅ exact match: this is the tokenizer that made tokens.bin")
    else:
        REUSE_TOKENS = False
        print("  ❌ no match: tokens.bin was made with a different tokenizer and can't be used.\n"
              "     The whole tar will be tokenized again with this tokenizer.")

if REUSE_TOKENS:
    assert Path(EXISTING_TOKENIZER).exists(), f"{EXISTING_TOKENIZER} not found: tokens.bin is useless without its tokenizer"
    if Path(EXISTING_TOKENIZER).resolve() != TOK_PATH.resolve():
        import shutil; shutil.copy(EXISTING_TOKENIZER, TOK_PATH)
    if USE_HF:                                          # later sessions must use this exact tokenizer too
        hf_api.upload_file(path_or_fileobj=str(TOK_PATH), path_in_repo="tokenizer.json", repo_id=HF_REPO)
    print(f"♻️  reusing {EXISTING_TOKENS_BIN} ({Path(EXISTING_TOKENS_BIN).stat().st_size/1e9:.1f} GB) "
          f"with its tokenizer {EXISTING_TOKENIZER}: the tar will NOT be tokenized again")

if not TOK_PATH.exists():
    hf_fetch("tokenizer.json", WORK_DIR)                   # same tokenizer as earlier sessions, if any

if TOK_PATH.exists():
    tokenizer = BPETokenizer.load(TOK_PATH)
    print(f"loaded saved tokenizer ({tokenizer.vocab_size} tokens) from {TOK_PATH}")
else:
    t0 = time.time()
    tokenizer = BPETokenizer().train(raw_text[:20_000_000], VOCAB_SIZE, verbose_every=1000)
    tokenizer.save(TOK_PATH)
    if USE_HF:
        hf_api.upload_file(path_or_fileobj=str(TOK_PATH), path_in_repo="tokenizer.json", repo_id=HF_REPO)
    print(f"\nTrained BPE with vocab size {tokenizer.vocab_size} in {time.time()-t0:.1f}s → {TOK_PATH}")

### See the tokenizer in action

Each `|` below is a token boundary. Common words become one token, and rare words are split into sub-word pieces.

In [ ]:
sample = "The quick brown fox jumps over the lazy dog. Extraordinarily, tokenization works!"
ids = tokenizer.encode(sample)
pieces = [tokenizer.decode([i]) for i in ids]

print("text   :", sample)
print("ids    :", ids)
print("tokens :", "|".join(pieces))
print(f"\n{len(sample)} characters → {len(ids)} tokens   (a char-level tokenizer would need {len(sample)})")
assert tokenizer.decode(ids) == sample, "round-trip failed!"
print("round trip OK ✓")

print("\nSome learned tokens:", [tokenizer.vocab[i].decode('utf-8', 'replace') for i in range(1000, 1030)])

### Encode the sample

We turn the sample books (and the validation books) into tensors of token ids. The sample is used for the demos below and for the validation loss. The **real training data** is tokenized on the fly while streaming (section 3b).

In [ ]:
t0 = time.time()
train_data = torch.tensor(tokenizer.encode(raw_text), dtype=torch.long)
val_data = torch.tensor(tokenizer.encode(val_text), dtype=torch.long)
chars_per_token = len(raw_text) / len(train_data)
print(f"sample: {len(raw_text):,} chars → {len(train_data):,} tokens ({chars_per_token:.2f} chars/token) in {time.time()-t0:.1f}s")
print(f"validation: {len(val_data):,} tokens (from books the model never trains on)")

lengths = [len(tokenizer.vocab[i]) for i in range(256, tokenizer.vocab_size - len(tokenizer.special))]
plt.figure(figsize=(6, 3))
plt.hist(lengths, bins=range(1, 16)); plt.xlabel("bytes per learned token"); plt.ylabel("count")
plt.title("How long are the BPE tokens?"); plt.show()

## 3. Dataset: input → target pairs 🎯

A language model is trained to **predict the next token**. From one stream of ids we cut windows of length `context_length`. The **target is the input shifted by one**:

```
 token stream:  [ 40, 367, 2885, 1464, 1807, 3619, 402, 271, ... ]

                  ┌─────────── context_length = 4 ────────────┐
 input  x:      [ 40,   367,  2885, 1464 ]
 target y:      [ 367,  2885, 1464, 1807 ]      ← same window, one step to the right

 This single window gives 4 training examples at once:
     [40]                  → 367
     [40, 367]             → 2885
     [40, 367, 2885]       → 1464
     [40, 367, 2885, 1464] → 1807
```

The window slides along the text by `stride` tokens. If `stride == context_length` the windows don't overlap.

In [ ]:
class GPTDataset(torch.utils.data.Dataset):
    def __init__(self, ids, context_length, stride):
        self.ids, self.T = ids, context_length
        self.starts = range(0, len(ids) - context_length - 1, stride)

    def __len__(self):
        return len(self.starts)

    def __getitem__(self, i):
        s = self.starts[i]
        chunk = self.ids[s : s + self.T + 1]
        return chunk[:-1], chunk[1:]          # x, y (y is x shifted by one)

def make_loader(ids, context_length, batch_size, stride=None, shuffle=True):
    ds = GPTDataset(ids, context_length, stride or context_length)
    return torch.utils.data.DataLoader(ds, batch_size=batch_size, shuffle=shuffle, drop_last=True)

# demo with a tiny context
demo_loader = make_loader(train_data, context_length=8, batch_size=2)
x, y = next(iter(demo_loader))
print("x:", x.shape, "\n", x)
print("y:", y.shape, "\n", y)
print("\nx[0] as text:", repr(tokenizer.decode(x[0].tolist())))
print("y[0] as text:", repr(tokenizer.decode(y[0].tolist())))

### 3b. Packaging our code for the background process 📦

Reading the whole tar and training take hours, so they run in a **separate background process** (`run.py`) instead of a notebook cell. That process can't see the notebook's variables, so we write the pieces it needs into two files:

* `llm_lib.py`: the tar streamer, the BPE tokenizer (encode/decode/load), and the GPT model. This is **the same code as above**, just compacted. The parameter names match, so weights move freely between the notebook and the script.
* `run.py`: the two long jobs:
  1. **Prepare.** Stream **every `.txt` file** of the tar in batches of 256 files, tokenize each batch in parallel on all CPU cores, and write the ids into **1 GB shards** (`tokens_00000.bin`, …). Each finished shard is uploaded to Hugging Face together with `progress.json` (the tar byte offset where the next shard starts).
  2. **Train.** Memory-map all shards and walk a random permutation of **every window of the corpus exactly once**. Every `CKPT_MINUTES` it saves and uploads a checkpoint.

```
 prepare:  tar ─stream─▶ [256 files] ─Pool─▶ ids ─▶ shard 0 (1 GB) ─upload─▶ HF   ─▶ shard 1 ─▶ … ─▶ end of tar ✅
 train:    shards (memory-mapped) ─▶ perm[0:B], perm[B:2B], … ─▶ GPU ─▶ checkpoint ─upload─▶ HF   ─▶ … ─▶ every window once ✅
 resume:   new runtime ─▶ download shards + progress + checkpoint ─▶ continue at the same byte / batch
```

The `%%writefile` cells below only *write* the files. They're launched in section 8.

In [ ]:
%%writefile llm_work/llm_lib.py
"""Shared code for the background process: tar streaming, BPE tokenizer, GPT model (same as the notebook)."""
import re, json, time, math, tarfile, urllib.request, http.client
import torch, torch.nn as nn, torch.nn.functional as F

HF_TAR_URL    = "https://huggingface.co/datasets/Lokeshlks/gutenberg_books_text/resolve/main/txt-files.tar"
DATASET_BYTES = 30_245_222_400
TAR_TRAILER   = 20_480                 # a tar ends with ≤ 20 KB of zero padding after the last file
EOT = "<|endoftext|>"

# ---------------------------------------------------------------- streaming the tar
def clean(text):
    text = text.replace("\r\n", "\n").replace("\r", "\n").replace("﻿", "")
    start = re.search(r"\*\*\* ?START OF (?:THE|THIS) PROJECT GUTENBERG.*?\*\*\*", text)
    end = re.search(r"\*\*\* ?END OF (?:THE|THIS) PROJECT GUTENBERG", text)
    if start:
        text = text[start.end():end.start() if end else None]
    return re.sub(r"\n{3,}", "\n\n", text).strip()

def stream_books(start_offset=0, english_only=False, max_retries=1000):
    """Yield (name, text, next_offset) for every .txt file in the tar, from a byte offset.

    Runs until there is no more data: until the read position reaches the END OF THE FILE.
    Hugging Face regularly cuts long downloads; each cut is resumed at the exact byte (HTTP Range)."""
    offset, retries = start_offset, 0
    while True:
        base = offset
        try:
            req = urllib.request.Request(HF_TAR_URL, headers={"Range": f"bytes={base}-"})
            with urllib.request.urlopen(req, timeout=60) as src, tarfile.open(fileobj=src, mode="r|") as tar:
                size = int(src.headers["Content-Range"].split("/")[-1])      # total size of the tar file
                for m in tar:
                    nxt = base + m.offset_data + (m.size + 511) // 512 * 512
                    if m.isfile() and m.name.lower().endswith(".txt"):
                        raw = tar.extractfile(m).read().decode("utf-8", errors="ignore")
                        if not english_only or re.search(r"^Language:\s*English", raw[:5000], re.M):
                            text = clean(raw)
                            if text:
                                yield m.name, text, nxt
                    offset = nxt
            if offset < size - TAR_TRAILER:
                # the connection was cut between two files and tarfile took it for the end of the archive
                raise EOFError(f"stream ended early at byte {offset:,} of {size:,}")
            return                                                  # reached the end of the file: no more data
        except (OSError, EOFError, tarfile.TarError, http.client.HTTPException) as e:
            retries += 1
            if retries > max_retries:
                raise
            print(f"  stream interrupted ({e!r}); resuming at byte {offset:,}", flush=True)
            time.sleep(min(60, 2 * retries))

# ---------------------------------------------------------------- BPE tokenizer (encode/decode/load)
PRETOKENIZE = re.compile(r"""'(?:[sdmt]|ll|ve|re)| ?[^\W\d_]+| ?\d{1,3}| ?[^\s\w]+| ?_+|\s+(?!\S)|\s+""")

def merge_ids(ids, pair, new_id):
    out, i = [], 0
    while i < len(ids):
        if i < len(ids) - 1 and ids[i] == pair[0] and ids[i + 1] == pair[1]:
            out.append(new_id); i += 2
        else:
            out.append(ids[i]); i += 1
    return out

class BPETokenizer:
    def __init__(self):
        self.merges, self.vocab, self.special, self._cache = {}, {i: bytes([i]) for i in range(256)}, {}, {}

    @classmethod
    def load(cls, path):
        d, tok = json.load(open(path)), cls()
        for a, b, n in d["merges"]:
            tok.merges[(a, b)] = n; tok.vocab[n] = tok.vocab[a] + tok.vocab[b]
        for sp, i in d["special"].items():
            tok.special[sp] = i; tok.vocab[i] = sp.encode("utf-8")
        tok._split = re.compile("(" + "|".join(re.escape(s) for s in tok.special) + ")")
        return tok

    @property
    def vocab_size(self):
        return len(self.vocab)

    def _encode_chunk(self, chunk):
        if chunk not in self._cache:
            if len(self._cache) > 500_000:
                self._cache.clear()
            ids = list(chunk.encode("utf-8"))
            while len(ids) >= 2:
                pair = min(zip(ids, ids[1:]), key=lambda p: self.merges.get(p, float("inf")))
                if pair not in self.merges:
                    break
                ids = merge_ids(ids, pair, self.merges[pair])
            self._cache[chunk] = ids
        return self._cache[chunk]

    def encode(self, text):
        ids = []
        for part in self._split.split(text):
            if part in self.special:
                ids.append(self.special[part])
            elif part:
                for chunk in PRETOKENIZE.findall(part):
                    ids.extend(self._encode_chunk(chunk))
        return ids

    def decode(self, ids):
        return b"".join(self.vocab[i] for i in ids).decode("utf-8", errors="replace")

# ---------------------------------------------------------------- GPT (same parameter names as the notebook)
class CausalSelfAttention(nn.Module):
    def __init__(self, d, n_heads, dropout):
        super().__init__()
        self.n_heads, self.dropout = n_heads, dropout
        self.qkv, self.proj = nn.Linear(d, 3 * d, bias=False), nn.Linear(d, d)
        self.resid_drop = nn.Dropout(dropout)

    def forward(self, x):
        B, T, C = x.shape
        q, k, v = (t.view(B, T, self.n_heads, C // self.n_heads).transpose(1, 2) for t in self.qkv(x).split(C, 2))
        y = F.scaled_dot_product_attention(q, k, v, is_causal=True, dropout_p=self.dropout if self.training else 0.0)
        return self.resid_drop(self.proj(y.transpose(1, 2).contiguous().view(B, T, C)))

class FeedForward(nn.Module):
    def __init__(self, d, dropout):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(d, 4 * d), nn.GELU(approximate="tanh"), nn.Linear(4 * d, d), nn.Dropout(dropout))

    def forward(self, x):
        return self.net(x)

class Block(nn.Module):
    def __init__(self, d, n_heads, dropout):
        super().__init__()
        self.ln1, self.ln2 = nn.LayerNorm(d), nn.LayerNorm(d)
        self.attn, self.ff = CausalSelfAttention(d, n_heads, dropout), FeedForward(d, dropout)

    def forward(self, x):
        x = x + self.attn(self.ln1(x))
        return x + self.ff(self.ln2(x))

class GPT(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.cfg = cfg
        d = cfg["d_model"]
        self.tok_emb, self.pos_emb = nn.Embedding(cfg["vocab_size"], d), nn.Embedding(cfg["context_length"], d)
        self.drop = nn.Dropout(cfg["dropout"])
        self.blocks = nn.ModuleList([Block(d, cfg["n_heads"], cfg["dropout"]) for _ in range(cfg["n_layers"])])
        self.ln_f = nn.LayerNorm(d)
        self.head = nn.Linear(d, cfg["vocab_size"], bias=False)
        self.head.weight = self.tok_emb.weight
        for m in self.modules():
            if isinstance(m, (nn.Linear, nn.Embedding)):
                nn.init.normal_(m.weight, 0.0, 0.02)
                if getattr(m, "bias", None) is not None:
                    nn.init.zeros_(m.bias)
        for name, p in self.named_parameters():
            if name.endswith("proj.weight") or name.endswith("net.2.weight"):
                nn.init.normal_(p, 0.0, 0.02 / math.sqrt(2 * cfg["n_layers"]))

    def forward(self, idx, targets=None):
        T = idx.shape[1]
        x = self.drop(self.tok_emb(idx) + self.pos_emb(torch.arange(T, device=idx.device)))
        for blk in self.blocks:
            x = blk(x)
        logits = self.head(self.ln_f(x))
        loss = None if targets is None else F.cross_entropy(logits.view(-1, logits.size(-1)), targets.reshape(-1))
        return logits, loss

In [ ]:
%%writefile llm_work/run.py
"""Background job: (1) stream + tokenize the WHOLE tar into shards, (2) train on every window once.
Everything is resumable, and progress is uploaded to Hugging Face so a new runtime can continue."""
import os, sys, json, time, math, random, fcntl, multiprocessing as mp
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"          # keep run.log readable
from array import array
from pathlib import Path
import torch
from llm_lib import *

W = Path(__file__).resolve().parent
C = json.load(open(W / "run_config.json"))
HF_REPO, HF_TOKEN = C["hf_repo"], os.environ.get("HF_TOKEN")
USE_HF = bool(HF_REPO and HF_TOKEN)
if USE_HF:
    from huggingface_hub import HfApi, hf_hub_download
    api = HfApi(token=HF_TOKEN)

def log(*a):
    print(time.strftime("%H:%M:%S"), *a, flush=True)

def status(**kw):
    (W / "status.json").write_text(json.dumps(dict(kw, time=time.time())))

def upload(path, retries=5):
    if not USE_HF:
        return
    for i in range(retries):
        try:
            api.upload_file(path_or_fileobj=str(path), path_in_repo=path.name, repo_id=HF_REPO)
            return
        except Exception as e:
            log(f"upload of {path.name} failed ({e!r}), retry {i+1}")
            time.sleep(10 * (i + 1))

def fetch(name):
    if (W / name).exists():
        return True
    if not USE_HF:
        return False
    try:
        hf_hub_download(HF_REPO, name, token=HF_TOKEN, local_dir=W)
        return True
    except Exception:
        return False

tok = BPETokenizer.load(W / "tokenizer.json")
EOT_ID = tok.special[EOT]

# =========================================================== 1) PREPARE: every .txt file → token shards
def _init(t):
    global _T
    _T = t

def _encode(text):
    return array("H", _T.encode(text) + [_T.special[EOT]]).tobytes()

FINGERPRINT = {"version": 3, "english_only": C["english_only"], "limit_bytes": C.get("limit_bytes"),
               "vocab_size": tok.vocab_size, "shard_tokens": C["shard_tokens"]}

def fresh_progress():
    return {"offset": 0, "files": 0, "tokens": 0, "text_bytes": 0, "shards": [], "done": False,
            "fingerprint": FINGERPRINT}

def use_existing_tokens():
    """Register an existing tokens.bin (whole tar, already tokenized) as the only shard: no re-tokenizing."""
    src = Path(C["existing_tokens"])
    n = src.stat().st_size // 2
    link = W / "tokens_existing.bin"
    if not (link.exists() and link.resolve() == src.resolve()):
        link.unlink(missing_ok=True)
        if src.parent.resolve() == W and src.name != link.name:
            src.rename(link)                                   # same folder: just rename it
        else:
            link.symlink_to(src)
    data = torch.from_file(str(link), shared=False, size=n, dtype=torch.int16)
    probe = data[:50_000_000].long()
    if int(probe.min()) < 0 or int(probe.max()) >= tok.vocab_size:
        raise RuntimeError("tokens.bin contains ids outside this tokenizer's vocabulary: wrong tokenizer.json?")
    n_docs = int((probe == EOT_ID).sum())
    log(f"♻️  using existing tokens: {n:,} tokens ({n*2/1e9:.1f} GB), no re-tokenizing")
    # yardstick measured on this tar with the 8,192-token tokenizer: ≈0.29 tokens per byte of tar → ≈8.8B tokens
    expected = 0.292 * DATASET_BYTES
    if tok.vocab_size == 8192 and n < 0.9 * expected and not C.get("limit_bytes"):
        log(f"⚠️  WARNING: the whole tar is ≈{expected/1e9:.1f}B tokens but this file has {n/1e9:.2f}B "
            f"({n/expected:.0%}). The old run probably stopped early (Hugging Face cuts long downloads). "
            f"Set EXISTING_TOKENS_BIN = None to tokenize the whole tar again.")
    else:
        log(f"    size check: ≈{n/expected:.0%} of the expected full-tar size ✓")
    log(f"    sanity check: ids in range ✓, {n_docs:,} <|endoftext|> in the first 50M tokens")
    i = random.Random(0).randrange(0, max(1, n - 200))
    log("    random sample: " + repr(tok.decode(data[i:i + 60].long().tolist()))[:200])
    for f in W.glob("tokens_0*.bin"):                          # partial shards of an interrupted re-tokenization
        f.unlink()
    P = {"offset": DATASET_BYTES, "files": None, "tokens": n, "text_bytes": None,
         "shards": [{"name": link.name, "tokens": n}], "done": True, "fingerprint": FINGERPRINT,
         "source": "existing tokens.bin"}
    (W / "progress.json").write_text(json.dumps(P))
    if C.get("upload_existing_tokens"):
        log(f"uploading {n*2/1e9:.1f} GB of tokens to Hugging Face (so a new runtime can resume)…")
        upload(link)
    upload(W / "progress.json")
    return P

def verify_shards(P):
    """Before training: every shard complete, ends on a file boundary, and #files == #<|endoftext|>."""
    total_eot = 0
    for s in P["shards"]:
        f = W / s["name"]
        if not f.exists() or f.stat().st_size != 2 * s["tokens"]:
            log(f"[check] {s['name']}: size {f.stat().st_size if f.exists() else 0:,} ≠ expected {2*s['tokens']:,} "
                f"→ re-downloading from Hugging Face")
            f.unlink(missing_ok=True)
            if not fetch(s["name"]) or f.stat().st_size != 2 * s["tokens"]:
                raise RuntimeError(f"{s['name']} is damaged locally and on Hugging Face → set RESET_DATA = True "
                                   f"in the config cell and relaunch to re-tokenize the tar cleanly")
        t = torch.from_file(str(f), shared=False, size=s["tokens"], dtype=torch.int16)
        if int(t[-1]) != EOT_ID:
            raise RuntimeError(f"{s['name']} doesn't end on a file boundary → set RESET_DATA = True and relaunch")
        n = sum(int((t[i:i + 100_000_000] == EOT_ID).sum()) for i in range(0, len(t), 100_000_000))
        total_eot += n
    log(f"[check] {len(P['shards'])} shards OK | {total_eot:,} documents (<|endoftext|>) vs {P['files'] or '?'} files")
    if P["files"] and total_eot != P["files"]:
        raise RuntimeError(f"shards hold {total_eot:,} documents but {P['files']:,} files were tokenized "
                           f"(duplicated or missing data) → set RESET_DATA = True and relaunch")

def prepare():
    if C.get("existing_tokens") and Path(C["existing_tokens"]).exists():
        return use_existing_tokens()
    fetch("progress.json")
    if C.get("reset_data"):                                    # one-time reset, identified by its id
        old = json.load(open(W / "progress.json")) if (W / "progress.json").exists() else {}
        if old.get("reset_id") != C["reset_data"]:
            log("RESET_DATA: deleting all token shards and progress → tokenizing the whole tar again from byte 0")
            for f in W.glob("tokens_*.bin"): f.unlink()
            for f in ["train_state.pt", "gpt_from_scratch.pt"]: (W / f).unlink(missing_ok=True)
            P0 = fresh_progress(); P0["reset_id"] = C["reset_data"]
            (W / "progress.json").write_text(json.dumps(P0)); upload(W / "progress.json")
    P = json.load(open(W / "progress.json")) if (W / "progress.json").exists() else fresh_progress()
    if P.get("fingerprint") != FINGERPRINT:
        log("⚠️  found progress.json from a run with DIFFERENT settings (e.g. an old demo run):",
            f"{P.get('files', 0):,} files, done={P.get('done')} → IGNORING it and starting the whole tar from byte 0")
        for f in W.glob("tokens_*.bin"): f.unlink()
        for f in ["train_state.pt", "gpt_from_scratch.pt"]: (W / f).unlink(missing_ok=True)
        P = fresh_progress()
        (W / "progress.json").write_text(json.dumps(P)); upload(W / "progress.json")
    if P["done"] and not C.get("limit_bytes") and P["offset"] < DATASET_BYTES - TAR_TRAILER:
        log(f"⚠️  progress.json says done but stopped at byte {P['offset']:,} of {DATASET_BYTES:,} → continuing from there")
        P["done"] = False
    for s in P["shards"]:                                   # make sure finished shards are on local disk
        if not fetch(s["name"]):
            raise RuntimeError(f"shard {s['name']} missing locally and on Hugging Face")
    if P["done"]:
        log(f"prepare already done: {P['files'] or 'all'} .txt files, "
            f"{P['tokens']:,} tokens in {len(P['shards'])} shards")
        return P
    if P["offset"]:
        log(f"resuming prepare at byte {P['offset']:,} ({P['offset']/DATASET_BYTES:.1%}), {P['files']:,} files done")

    limit = C.get("limit_bytes")
    t0, off0 = time.time(), P["offset"]
    shard_name = lambda k: f"tokens_{k:05d}.bin"
    out, cur_tokens, pend = open(W / shard_name(len(P["shards"])), "wb"), 0, {"files": 0, "text_bytes": 0}

    def close_shard(next_offset):
        nonlocal out, cur_tokens, pend
        out.close()
        name = shard_name(len(P["shards"]))
        if cur_tokens:
            upload(W / name)
            P["shards"].append({"name": name, "tokens": cur_tokens})
        P.update(offset=next_offset, files=P["files"] + pend["files"], tokens=P["tokens"] + cur_tokens,
                 text_bytes=P["text_bytes"] + pend["text_bytes"])
        (W / "progress.json").write_text(json.dumps(P)); upload(W / "progress.json")
        out, cur_tokens, pend = open(W / shard_name(len(P["shards"])), "wb"), 0, {"files": 0, "text_bytes": 0}

    with mp.get_context("fork").Pool(os.cpu_count() or 2, initializer=_init, initargs=(tok,)) as pool:
        batch, nb, last = [], 0, P["offset"]
        for name, text, nxt in stream_books(P["offset"], english_only=C["english_only"]):
            batch.append(text); last = nxt
            if len(batch) < C["files_per_batch"] and not (limit and nxt >= limit):
                continue
            for blob in pool.map(_encode, batch, chunksize=4):
                out.write(blob); cur_tokens += len(blob) // 2
            pend["files"] += len(batch); pend["text_bytes"] += sum(map(len, batch)); batch = []; nb += 1
            if cur_tokens >= C["shard_tokens"]:
                close_shard(nxt)
            if nb % 10 == 0:
                speed = (nxt - off0) / (time.time() - t0)
                eta = (DATASET_BYTES - nxt) / max(speed, 1) / 3600
                log(f"[prepare] {nxt/DATASET_BYTES:6.2%} of tar | {P['files'] + pend['files']:8,} files | "
                    f"{(P['tokens'] + cur_tokens)/1e9:5.2f}B tokens | {speed/1e6:5.1f} MB/s | ETA {eta:4.1f} h")
                status(phase="prepare", pct=nxt / DATASET_BYTES, files=P["files"] + pend["files"])
            if limit and nxt >= limit:
                log(f"(demo limit of {limit/1e6:.0f} MB reached)")
                break
        if batch:
            for blob in pool.map(_encode, batch, chunksize=4):
                out.write(blob); cur_tokens += len(blob) // 2
            pend["files"] += len(batch); pend["text_bytes"] += sum(map(len, batch))
    close_shard(last)
    out.close(); (W / shard_name(len(P["shards"]))).unlink(missing_ok=True)       # empty trailing shard
    if not limit and P["offset"] < DATASET_BYTES - TAR_TRAILER:
        raise RuntimeError(f"stream stopped at byte {P['offset']:,} of {DATASET_BYTES:,}; re-launch to continue")
    P["done"] = True
    (W / "progress.json").write_text(json.dumps(P)); upload(W / "progress.json")
    log(f"✅ {'WHOLE TAR' if not limit else 'DEMO SLICE'} PROCESSED: {P['files']:,} .txt files | {P['text_bytes']/1e9:.1f} GB text | "
        f"{P['tokens']:,} tokens in {len(P['shards'])} shards")
    return P

# =========================================================== 2) TRAIN: every window of every shard, once
def pick_batch_and_lr():
    """Batch size scaled to GPU memory (a bigger batch keeps the GPU busy), LR scaled with sqrt(batch)."""
    B, lr = C["batch_size"], C["lr"]
    if B == "auto":
        gb = torch.cuda.get_device_properties(0).total_memory / 1e9 if torch.cuda.is_available() else 0
        B = 512 if gb >= 70 else 256 if gb >= 35 else 128 if gb >= 20 else 64
    if lr == "auto":
        lr = min(1.5e-3, 6e-4 * math.sqrt(B / 64))
    return B, lr

def train(P):
    import threading, queue
    cfg, T = C["cfg"], C["cfg"]["context_length"]
    B, LR = pick_batch_and_lr()
    device = "cuda" if torch.cuda.is_available() else "cpu"
    use_amp = device == "cuda"
    if use_amp:
        torch.backends.cuda.matmul.allow_tf32 = True
        torch.backends.cudnn.allow_tf32 = True

    # ---- data: all shards; loaded into RAM if they fit (much faster random access than disk)
    parts = [torch.from_file(str(W / s["name"]), shared=False, size=s["tokens"], dtype=torch.int16) for s in P["shards"]]
    need = sum(len(p) for p in parts) * 2
    free = os.sysconf("SC_AVPHYS_PAGES") * os.sysconf("SC_PAGE_SIZE")
    if need < 0.7 * free:
        t0 = time.time()
        parts = [p.clone() for p in parts]
        log(f"[train] loaded all {need/1e9:.1f} GB of tokens into RAM ({free/1e9:.0f} GB free) in {time.time()-t0:.0f}s")
    else:
        log(f"[train] tokens ({need/1e9:.1f} GB) read from disk via memory-map (only {free/1e9:.0f} GB RAM free)")
    val = parts[-1][-C["val_tokens"]:]
    parts[-1] = parts[-1][:-C["val_tokens"]]                                   # held-out tail
    counts = torch.tensor([(len(p) - 1) // T for p in parts])
    ends = counts.cumsum(0)
    n_windows = int(ends[-1])
    perm = torch.randperm(n_windows, generator=torch.Generator().manual_seed(C["seed"]))
    total = n_windows // B if not C.get("max_steps") else min(C["max_steps"], n_windows // B)

    def make_batch(i):
        idx = perm[i * B:(i + 1) * B]
        sh = torch.searchsorted(ends, idx, right=True)                          # which shard
        local = idx - (ends[sh] - counts[sh])                                   # window inside that shard
        w = torch.stack([parts[s][l * T:l * T + T + 1] for s, l in zip(sh.tolist(), local.tolist())]).long()
        return w.pin_memory() if use_amp else w

    # ---- model, compiled; fused AdamW
    model = GPT(cfg).to(device)
    decay = [p for p in model.parameters() if p.dim() >= 2]
    no_decay = [p for p in model.parameters() if p.dim() < 2]
    opt = torch.optim.AdamW([{"params": decay, "weight_decay": 0.1}, {"params": no_decay, "weight_decay": 0.0}],
                            lr=LR, betas=(0.9, 0.95), **({"fused": True} if use_amp else {}))
    amp_dtype = torch.bfloat16 if use_amp and torch.cuda.is_bf16_supported() else torch.float16
    scaler = torch.amp.GradScaler("cuda", enabled=use_amp and amp_dtype == torch.float16)

    step, history = 0, {"step": [], "train": [], "val": []}
    if fetch("train_state.pt"):
        st = torch.load(W / "train_state.pt", map_location=device, weights_only=False)
        if st["cfg"] == cfg and st.get("data") == [P["files"], P["tokens"]] and st.get("batch_size") == B:
            model.load_state_dict(st["model"]); opt.load_state_dict(st["optimizer"]); scaler.load_state_dict(st["scaler"])
            step, history = st["step"], st["history"]
            log(f"resuming training at batch {step:,}/{total:,} ({step/total:.1%})")
        else:
            log("found a checkpoint made with a different model/data/batch size → training from scratch")

    fmodel = model
    if use_amp and C.get("compile", True):
        try:
            fmodel = torch.compile(model)
            log("[train] torch.compile enabled (first steps are slow while it compiles)")
        except Exception as e:
            log(f"[train] torch.compile unavailable ({e!r}); running eagerly")

    def lr_at(s):
        if s < C["warmup"]:
            return LR * (s + 1) / C["warmup"]
        prog = (s - C["warmup"]) / max(1, total - C["warmup"])
        return LR * (0.1 + 0.9 * 0.5 * (1 + math.cos(math.pi * min(prog, 1.0))))

    val_starts = torch.randint(len(val) - T - 1, (C["eval_windows"],), generator=torch.Generator().manual_seed(0)).tolist()

    @torch.no_grad()
    def val_loss():
        model.eval(); tot = torch.zeros((), device=device); n = 0
        for i in range(0, len(val_starts), B):
            w = torch.stack([val[j:j + T + 1] for j in val_starts[i:i + B]]).long().to(device)
            with torch.autocast("cuda", dtype=amp_dtype, enabled=use_amp):
                tot += model(w[:, :-1], w[:, 1:])[1].float(); n += 1
        model.train(); return (tot / n).item()

    @torch.no_grad()
    def sample(prompt="The", n=40):
        model.eval()
        x = torch.tensor([tok.encode(prompt)], device=device)
        for _ in range(n):
            logits = model(x[:, -T:])[0][:, -1].float() / 0.8
            v, _ = torch.topk(logits, 50); logits[logits < v[:, [-1]]] = -float("inf")
            x = torch.cat([x, torch.multinomial(torch.softmax(logits, -1), 1)], 1)
        model.train(); return tok.decode(x[0].tolist()).replace("\n", " ")

    def checkpoint():
        torch.save({"model": model.state_dict(), "optimizer": opt.state_dict(), "scaler": scaler.state_dict(),
                    "cfg": cfg, "step": step, "history": history, "data": [P["files"], P["tokens"]], "batch_size": B},
                   W / "train_state.tmp")
        (W / "train_state.tmp").replace(W / "train_state.pt")
        torch.save({"model": model.state_dict(), "cfg": cfg, "step": step, "tokens_seen": step * B * T},
                   W / "gpt_from_scratch.pt")
        upload(W / "train_state.pt"); upload(W / "gpt_from_scratch.pt")

    # ---- background thread prepares the next batches while the GPU trains on the current one
    q = queue.Queue(maxsize=8)
    def producer(start):
        for i in range(start, total):
            q.put(make_batch(i))
    threading.Thread(target=producer, args=(step,), daemon=True).start()

    log(f"[train] data: {P['files'] or '(existing tokens.bin)'} files, {P['tokens']:,} tokens, {len(parts)} shards")
    log(f"[train] batch size {B} (auto-scaled to GPU memory), lr {LR:.2e}: "
        f"{n_windows:,} windows → {total:,} batches of {B}x{T} (one full pass)")
    t0, last_ckpt, step0 = time.time(), time.time(), step
    loss_sum, loss_n = torch.zeros((), device=device), 0
    model.train()
    while step < total:
        for gr in opt.param_groups:
            gr["lr"] = lr_at(step)
        w = q.get().to(device, non_blocking=True)
        try:
            with torch.autocast("cuda", dtype=amp_dtype, enabled=use_amp):
                _, loss = fmodel(w[:, :-1], w[:, 1:])
        except Exception as e:
            if fmodel is model:
                raise
            log(f"[train] torch.compile failed at run time ({type(e).__name__}); falling back to eager mode")
            fmodel = model
            with torch.autocast("cuda", dtype=amp_dtype, enabled=use_amp):
                _, loss = fmodel(w[:, :-1], w[:, 1:])
        opt.zero_grad(set_to_none=True)
        scaler.scale(loss).backward()
        scaler.unscale_(opt); torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(opt); scaler.update()
        loss_sum += loss.detach().float(); loss_n += 1; step += 1                 # no GPU→CPU sync per step
        if step % C["eval_every"] == 0 or step == total:
            vl = val_loss()
            history["step"].append(step); history["train"].append((loss_sum / loss_n).item()); history["val"].append(vl)
            loss_sum.zero_(); loss_n = 0
            tok_s = (step - step0) * B * T / (time.time() - t0)
            eta = (total - step) * B * T / tok_s / 3600
            mem = f" | GPU mem {torch.cuda.max_memory_allocated()/1e9:4.1f} GB" if use_amp else ""
            log(f"[train] batch {step:,}/{total:,} ({step/total:6.2%} of corpus) | train {history['train'][-1]:.3f} | "
                f"val {vl:.3f} | {tok_s/1e3:5.0f}k tok/s{mem} | ETA {eta:4.1f} h")
            log("   ↳ " + sample()[:160])
            status(phase="train", pct=step / total, step=step, total=total)
        first_report = step == step0 + C["eval_every"]                       # save something early, then every N min
        if first_report or time.time() - last_ckpt > C["ckpt_minutes"] * 60:
            checkpoint(); last_ckpt = time.time()
    checkpoint()
    log(f"🎉 TRAINING DONE: {step:,} batches = {step*B*T:,} tokens, every window of the corpus seen once")

if __name__ == "__main__":
    lock = open(W / "run.lock", "w")
    try:
        fcntl.flock(lock, fcntl.LOCK_EX | fcntl.LOCK_NB)          # only ONE run.py at a time
    except BlockingIOError:
        log("another run.py is already running → this one exits (use the monitor cell to watch it)")
        sys.exit(0)
    status(phase="starting", pct=0)
    log("MODE:", "FULL RUN (every .txt file of the whole tar, one full training pass)" if not C.get("limit_bytes")
        else f"DEMO (first {C['limit_bytes']/1e6:.0f} MB only)", "| device:", "cuda" if torch.cuda.is_available() else "CPU")
    P = prepare()
    verify_shards(P)
    train(P)
    status(phase="done", pct=1.0)

## 4. Embeddings: from ids to vectors 🔢➡️📐

A token id like `2885` has no meaning by itself. An **embedding layer** is a lookup table with one learnable row (vector) per token:

```
   token embedding table (vocab_size × d_model)       positional embedding table (context_length × d_model)
          ┌──────────────────────────────┐                   ┌──────────────────────────────┐
     0    │ 0.12 -0.40  0.88 ...  0.05   │              0    │ ...                          │
     1    │ ...                          │              1    │ ...                          │
     ⋮    │                              │              ⋮    │                              │
  2885 ──▶│ 0.31  0.07 -1.20 ...  0.66   │         pos p ──▶ │ 0.02 -0.11  0.45 ... -0.30   │
     ⋮    │                              │              ⋮    │                              │
          └──────────────────────────────┘                   └──────────────────────────────┘

   x = token_embedding[id]  +  position_embedding[position]        → shape (batch, T, d_model)
```

Attention by itself has **no sense of order**: "dog bites man" and "man bites dog" would look identical. So we **add** a learned position vector. During training, similar tokens end up with similar vectors.

In [ ]:
d_demo = 16
tok_emb = nn.Embedding(tokenizer.vocab_size, d_demo)
pos_emb = nn.Embedding(8, d_demo)

xb, _ = next(iter(demo_loader))                     # (B=2, T=8) ids
te = tok_emb(xb)                                    # (2, 8, 16)
pe = pos_emb(torch.arange(xb.shape[1]))             # (8, 16), broadcast over batch
h = te + pe
print("ids", tuple(xb.shape), "→ token emb", tuple(te.shape), "+ pos emb", tuple(pe.shape), "→", tuple(h.shape))

## 5. Self-attention, step by step 👀

This is the heart of the Transformer. Each token asks *"which earlier tokens are relevant to me?"* and builds a new vector as a **weighted average** of them.

Each token vector `x` is projected into three vectors:

| | Meaning | Analogy |
|---|---|---|
| **Query** `q = x·W_q` | what I'm looking for | a search query |
| **Key** `k = x·W_k` | what I contain | the title of a document |
| **Value** `v = x·W_v` | what I'll hand over if chosen | the document's content |

```
          ┌─────┐
     ┌───▶│ W_q │──▶ Q ──┐
     │    └─────┘        │
     │    ┌─────┐        │     ┌──────────────┐    ┌─────────────┐    ┌─────────┐
  X ─┼───▶│ W_k │──▶ K ──┴───▶ │ Q·Kᵀ / √d    │──▶ │ causal mask │──▶ │ softmax │──▶ A (T×T)
     │    └─────┘              │ scores (T×T) │    │ future=-inf │    └─────────┘      │
     │    ┌─────┐              └──────────────┘    └─────────────┘                     ▼
     └───▶│ W_v │──▶ V ─────────────────────────────────────────────────────────▶  out = A · V
          └─────┘
```

**Causal mask.** When predicting token *t* the model must not peek at tokens *t+1, t+2, ...*, so we set those scores to `-inf` before the softmax (and they become 0):

```
            keys →   The  cat  sat  on
 queries ↓  The    [  ✓    ✗    ✗    ✗ ]
            cat    [  ✓    ✓    ✗    ✗ ]
            sat    [  ✓    ✓    ✓    ✗ ]
            on     [  ✓    ✓    ✓    ✓ ]
```

Let's do it by hand on a tiny example.

In [ ]:
torch.manual_seed(0)
words = ["The", " cat", " sat", " on", " the", " mat"]
T, d = len(words), 8
X = torch.randn(T, d)                        # pretend these are embeddings for 6 tokens

W_q, W_k, W_v = (torch.randn(d, d) for _ in range(3))
Q, K, V = X @ W_q, X @ W_k, X @ W_v          # (T, d) each

# 1) similarity of every query with every key
scores = Q @ K.T / math.sqrt(d)              # (T, T), scaled so softmax isn't too peaky
# 2) causal mask: no looking into the future
mask = torch.triu(torch.ones(T, T, dtype=torch.bool), diagonal=1)
scores = scores.masked_fill(mask, float("-inf"))
# 3) softmax → each row sums to 1
A = torch.softmax(scores, dim=-1)
# 4) weighted sum of values
out = A @ V                                   # (T, d)

print("row sums:", A.sum(-1))
fig, ax = plt.subplots(figsize=(5, 4))
im = ax.imshow(A.detach(), cmap="viridis")
ax.set_xticks(range(T), words, rotation=45); ax.set_yticks(range(T), words)
ax.set_xlabel("key (attended to)"); ax.set_ylabel("query (attending)")
ax.set_title("Causal attention weights (random, untrained)")
plt.colorbar(im); plt.show()

### Multi-head attention

One attention "head" can focus on only one kind of relationship at a time. **Multi-head attention** runs `n_heads` smaller attentions in parallel, for example one head tracking grammar and another tracking which name "she" refers to. It then concatenates the results and mixes them with an output projection.

```
 X ─▶ Linear(d, 3d) ─▶ split into Q, K, V ─▶ reshape to (B, n_heads, T, head_dim)
                                                │  head 1: softmax(QKᵀ/√hd + mask)·V
                                                │  head 2: ...
                                                │  head h: ...
                                                ▼
                     concat heads (B, T, d) ─▶ Linear(d, d) ─▶ dropout ─▶ out
```

We implement the math manually. `F.scaled_dot_product_attention` (also plain PyTorch) computes exactly the same thing with a fused, faster kernel, and we switch to it when `use_fused=True`.

In [ ]:
class CausalSelfAttention(nn.Module):
    def __init__(self, d_model, n_heads, context_length, dropout=0.0, use_fused=True):
        super().__init__()
        assert d_model % n_heads == 0
        self.n_heads, self.head_dim = n_heads, d_model // n_heads
        self.qkv = nn.Linear(d_model, 3 * d_model, bias=False)   # W_q, W_k, W_v in one matmul
        self.proj = nn.Linear(d_model, d_model)
        self.attn_drop, self.resid_drop = nn.Dropout(dropout), nn.Dropout(dropout)
        self.dropout, self.use_fused = dropout, use_fused
        self.register_buffer("mask", torch.triu(torch.ones(context_length, context_length, dtype=torch.bool), 1),
                             persistent=False)

    def forward(self, x, return_attn=False):
        B, T, C = x.shape
        q, k, v = self.qkv(x).split(C, dim=2)
        # (B, T, C) -> (B, n_heads, T, head_dim)
        q, k, v = (t.view(B, T, self.n_heads, self.head_dim).transpose(1, 2) for t in (q, k, v))

        if self.use_fused and not return_attn:
            y = F.scaled_dot_product_attention(q, k, v, is_causal=True,
                                               dropout_p=self.dropout if self.training else 0.0)
            att = None
        else:
            att = (q @ k.transpose(-2, -1)) / math.sqrt(self.head_dim)   # (B, h, T, T)
            att = att.masked_fill(self.mask[:T, :T], float("-inf"))
            att = torch.softmax(att, dim=-1)
            y = self.attn_drop(att) @ v                                  # (B, h, T, head_dim)

        y = y.transpose(1, 2).contiguous().view(B, T, C)                 # concat heads
        y = self.resid_drop(self.proj(y))
        return (y, att) if return_attn else y

mha = CausalSelfAttention(d_model=16, n_heads=4, context_length=8)
print("in", tuple(h.shape), "→ out", tuple(mha(h).shape))

# sanity check: manual math == fused kernel
mha.eval()
y_manual, _ = mha(h, return_attn=True)
y_fused = mha(h)
print("manual vs fused max diff:", (y_manual - y_fused).abs().max().item())

## 6. The Transformer block and the full GPT 🏗️

### Building blocks

* **LayerNorm** rescales each token vector to mean 0 and variance 1 (plus a learnable scale and shift). This keeps training stable.
* **Feed-forward (MLP)**: `Linear(d → 4d) → GELU → Linear(4d → d)`, applied to each token independently. Attention *mixes information between tokens*; the MLP *processes* it within each token.
* **Residual connections** (`x = x + f(x)`) give gradients a "highway" through deep networks.

### One Transformer block (pre-LayerNorm, as in GPT-2)

```
        x ─────────────────────────┐                       x ───────────────────────┐
        │                          │                       │                        │
        ▼                          │                       ▼                        │
   ┌──────────┐   ┌────────────┐   │   (+)            ┌──────────┐   ┌─────────┐    │   (+)
   │LayerNorm │──▶│ Multi-Head │───┴──▶ ● ──────────▶ │LayerNorm │──▶│   MLP   │────┴──▶ ● ──▶ out
   └──────────┘   │ Attention  │                      └──────────┘   │(4× GELU)│
                  └────────────┘                                     └─────────┘
```

### The whole GPT

```
  token ids (B, T)
        │
        ▼
  ┌────────────────────────┐  ┌──────────────────────────┐
  │ Token embedding (V×d)  │ +│ Position embedding (T×d) │
  └────────────────────────┘  └──────────────────────────┘
        │  dropout
        ▼
  ┌────────────────────────┐
  │  Transformer block 1   │
  │  Transformer block 2   │   × n_layers
  │          ...           │
  │  Transformer block N   │
  └────────────────────────┘
        │
        ▼
    Final LayerNorm
        │
        ▼
  Linear head (d → V)   ← shares weights with the token embedding ("weight tying")
        │
        ▼
  logits (B, T, V)  →  softmax  →  probability of every possible next token
```

In [ ]:
class LayerNorm(nn.Module):
    """LayerNorm written out by hand (same as nn.LayerNorm)."""
    def __init__(self, d, eps=1e-5):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(d))
        self.bias = nn.Parameter(torch.zeros(d))

    def forward(self, x):
        mean = x.mean(-1, keepdim=True)
        var = x.var(-1, keepdim=True, unbiased=False)
        return self.weight * (x - mean) / torch.sqrt(var + self.eps) + self.bias


class GELU(nn.Module):
    """Gaussian Error Linear Unit (tanh approximation used by GPT-2)."""
    def forward(self, x):
        return 0.5 * x * (1 + torch.tanh(math.sqrt(2 / math.pi) * (x + 0.044715 * x ** 3)))


class FeedForward(nn.Module):
    def __init__(self, d_model, dropout):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(d_model, 4 * d_model), GELU(),
                                 nn.Linear(4 * d_model, d_model), nn.Dropout(dropout))

    def forward(self, x):
        return self.net(x)


class TransformerBlock(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.ln1 = LayerNorm(cfg["d_model"])
        self.attn = CausalSelfAttention(cfg["d_model"], cfg["n_heads"], cfg["context_length"],
                                        cfg["dropout"], cfg.get("use_fused", True))
        self.ln2 = LayerNorm(cfg["d_model"])
        self.ff = FeedForward(cfg["d_model"], cfg["dropout"])

    def forward(self, x):
        x = x + self.attn(self.ln1(x))     # residual 1
        x = x + self.ff(self.ln2(x))       # residual 2
        return x


# visualize GELU vs ReLU
xs = torch.linspace(-4, 4, 200)
plt.figure(figsize=(5, 3))
plt.plot(xs, GELU()(xs), label="GELU"); plt.plot(xs, F.relu(xs), "--", label="ReLU")
plt.legend(); plt.grid(alpha=.3); plt.title("GELU is a smooth ReLU"); plt.show()

In [ ]:
class GPT(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.cfg = cfg
        self.tok_emb = nn.Embedding(cfg["vocab_size"], cfg["d_model"])
        self.pos_emb = nn.Embedding(cfg["context_length"], cfg["d_model"])
        self.drop = nn.Dropout(cfg["dropout"])
        self.blocks = nn.ModuleList([TransformerBlock(cfg) for _ in range(cfg["n_layers"])])
        self.ln_f = LayerNorm(cfg["d_model"])
        self.head = nn.Linear(cfg["d_model"], cfg["vocab_size"], bias=False)
        self.head.weight = self.tok_emb.weight          # weight tying (saves V×d parameters)

        self.apply(self._init_weights)
        # GPT-2 trick: scale down the last projection of each residual branch
        for name, p in self.named_parameters():
            if name.endswith("proj.weight") or name.endswith("net.2.weight"):
                nn.init.normal_(p, mean=0.0, std=0.02 / math.sqrt(2 * cfg["n_layers"]))

    def _init_weights(self, m):
        if isinstance(m, nn.Linear):
            nn.init.normal_(m.weight, mean=0.0, std=0.02)
            if m.bias is not None:
                nn.init.zeros_(m.bias)
        elif isinstance(m, nn.Embedding):
            nn.init.normal_(m.weight, mean=0.0, std=0.02)

    def forward(self, idx, targets=None):
        B, T = idx.shape
        assert T <= self.cfg["context_length"], "sequence longer than context_length"
        pos = torch.arange(T, device=idx.device)
        x = self.drop(self.tok_emb(idx) + self.pos_emb(pos))     # (B, T, d)
        for block in self.blocks:
            x = block(x)
        logits = self.head(self.ln_f(x))                         # (B, T, vocab)

        loss = None
        if targets is not None:
            loss = F.cross_entropy(logits.view(-1, logits.size(-1)), targets.view(-1))
        return logits, loss

    def num_params(self):
        return sum(p.numel() for p in self.parameters())

## 7. Pick a model size and a training plan ⚙️

| Hyperparameter | Meaning |
|---|---|
| `context_length` | how many tokens the model can "see" at once |
| `d_model` | size of every token vector |
| `n_heads` | number of attention heads (must divide `d_model`) |
| `n_layers` | number of stacked Transformer blocks |
| `dropout` | randomly zeroes activations during training. We have so much data that we never repeat a book, so it's set to 0 |

**The training plan: one pass over the whole tar.** The background job trains until **every window of every token shard has been seen exactly once**, so the full 30 GB. The learning-rate schedule is laid out over exactly that pass.

How long it takes depends on the GPU. The default model (~11M parameters) processes very roughly 100–150k tokens/s on a T4 and several times that on an A100 or L4, so ≈8B tokens takes roughly 15–20 h on a T4 and ~4–6 h on an A100. Before that comes the one-time tokenization of the tar (~1–2 h). Everything is checkpointed to Hugging Face, so it can span several Colab sessions.

This cell writes all settings to `run_config.json`, which is what `run.py` reads.

On CPU the plan shrinks to a 300-step demo.

In [ ]:
if device == "cuda":
    cfg = dict(vocab_size=tokenizer.vocab_size, context_length=256, d_model=384, n_heads=6, n_layers=6, dropout=0.0)
    B, LR, WARMUP = 64, 6e-4, 1000
    EVAL_EVERY, CKPT_MINUTES = 1000, 20
else:   # CPU / Apple MPS: a toy model so it still runs
    cfg = dict(vocab_size=tokenizer.vocab_size, context_length=128, d_model=128, n_heads=4, n_layers=4, dropout=0.0)
    B, LR, WARMUP = 16, 1e-3, 30
    EVAL_EVERY, CKPT_MINUTES = 50, 10

# FULL_RUN = True  → run.py tokenizes EVERY .txt file of the tar and trains one full pass over all of it.
# FULL_RUN = False → quick demo (first 20 MB of the tar, 300 steps), only for testing the pipeline.
FULL_RUN = True
# RESET_DATA = True → throw away all token shards and re-tokenize the whole tar from byte 0 (~40 min).
# Only needed if the data check in run.log reports damaged/inconsistent shards. Set back to False afterwards.
RESET_DATA = False
if FULL_RUN:
    print("✅ FULL RUN: the whole 30 GB tar will be tokenized and trained on")
    if device != "cuda":
        print("⚠️  No GPU detected! Runtime → Change runtime type → GPU (A100/L4/T4). On CPU this would take weeks.")
else:
    print("🧪 DEMO RUN: only the first 20 MB of the tar (set FULL_RUN = True for everything)")

T = cfg["context_length"]
tokens_per_step = B * T
est_tokens = DATASET_BYTES * 0.9 / chars_per_token                   # rough: the real count comes from run.py
TOTAL_STEPS = int(est_tokens / tokens_per_step)
print(f"whole tar ≈ {est_tokens/1e9:.1f}B tokens ≈ {TOTAL_STEPS:,} batches of {tokens_per_step:,} tokens (estimate)")

run_config = {
    "cfg": cfg,
    "batch_size": "auto" if device == "cuda" else B,   # auto = scaled to GPU memory (64 on T4 … 512 on A100-80GB)
    "lr": "auto" if device == "cuda" else LR,           # auto = scaled with sqrt(batch size)
    "compile": True,                                    # torch.compile for more GPU throughput
    "reset_data": time.time() if RESET_DATA else None,   # a one-time reset id (a relaunch won't reset again) "warmup": WARMUP, "eval_every": EVAL_EVERY, "ckpt_minutes": CKPT_MINUTES,
    "eval_windows": 50 * B if device == "cuda" else 10 * B, "val_tokens": 2_000_000 if device == "cuda" else 50_000,
    "seed": SEED, "english_only": ENGLISH_ONLY, "files_per_batch": 256,
    "shard_tokens": 500_000_000 if device == "cuda" else 1_000_000,     # 1 GB shards (2 bytes per token)
    "limit_bytes": None if FULL_RUN else 20_000_000,                  # None = the WHOLE tar
    "max_steps": None if FULL_RUN else 300,                           # None = one full pass over everything
    "hf_repo": HF_REPO,
    "existing_tokens": str(Path(EXISTING_TOKENS_BIN).resolve()) if REUSE_TOKENS else None,
    "upload_existing_tokens": True,        # copy tokens.bin to your HF repo so a recycled runtime can resume
}
(WORK_DIR / "run_config.json").write_text(json.dumps(run_config, indent=1))

torch.manual_seed(SEED)
model = GPT(cfg).to(device)
print(json.dumps(cfg, indent=1))
print(f"\nParameters: {model.num_params()/1e6:.2f} M")
for name, mod in [("embeddings", model.tok_emb), ("pos emb", model.pos_emb), ("one block", model.blocks[0])]:
    print(f"  {name:<11} {sum(p.numel() for p in mod.parameters()):>10,}")

### Before training: the model is random

The expected loss for random guessing is `ln(vocab_size)`. Let's check we're close to it, and look at what an untrained model "writes".

In [ ]:
@torch.no_grad()
def generate(model, idx, max_new_tokens, temperature=1.0, top_k=None):
    """Autoregressive sampling: predict → sample → append → repeat."""
    model.eval()
    for _ in range(max_new_tokens):
        idx_cond = idx[:, -model.cfg["context_length"]:]         # crop to the context window
        logits, _ = model(idx_cond)
        logits = logits[:, -1, :] / max(temperature, 1e-6)       # only the last position matters
        if top_k is not None:
            v, _ = torch.topk(logits, min(top_k, logits.size(-1)))
            logits[logits < v[:, [-1]]] = -float("inf")          # keep only the top-k candidates
        probs = torch.softmax(logits, dim=-1)
        next_id = torch.multinomial(probs, num_samples=1)
        idx = torch.cat([idx, next_id], dim=1)
    return idx

def complete(prompt, max_new_tokens=80, temperature=0.8, top_k=50):
    idx = torch.tensor([tokenizer.encode(prompt)], dtype=torch.long, device=device)
    return tokenizer.decode(generate(model, idx, max_new_tokens, temperature, top_k)[0].tolist())

xb, yb = next(iter(make_loader(train_data, cfg["context_length"], 4)))
_, loss = model(xb.to(device), yb.to(device))
print(f"initial loss {loss.item():.3f}   vs   ln(vocab) = {math.log(cfg['vocab_size']):.3f}")
print("\nUntrained sample:\n", complete("It was a dark and stormy night", 40))

## 8. Training in the background 🏋️

The training loop inside `run.py` is the same loop every neural network uses:

```
   ┌──────────────────────────────────────────────────────────────────────┐
   │ 1. get batch (x, y)          ← next B windows of the permutation     │
   │ 2. forward:   logits = model(x)                                      │
   │ 3. loss:      cross_entropy(logits, y)   "how surprised were we?"    │
   │ 4. backward:  loss.backward()            gradients for all weights   │
   │ 5. step:      optimizer.step()           nudge weights downhill      │
   └────────────────────────────── repeat ────────────────────────────────┘
```

It also uses these extras from real LLM training:
* **AdamW** with weight decay,
* **warm-up + cosine learning-rate decay** over the whole pass,
* **gradient clipping**,
* **mixed precision**,
* **checkpoints** every `CKPT_MINUTES`, uploaded to Hugging Face.

**How to use these cells:**

| Situation | What to do |
|---|---|
| First time | Run the launch cell. It starts `run.py` (prepare → train) in the background. |
| Page reloaded / tab closed | Nothing stopped. Just run the **monitor** cell again. |
| Runtime recycled / new session | **Run all.** Setup downloads the tokenizer, and the launch cell restarts `run.py`, which downloads the shards and checkpoint and continues. |
| Want to stop | Run the stop cell (progress up to the last checkpoint is kept). |

In [ ]:
def lr_at(step, max_lr, warmup, max_steps, min_ratio=0.1):
    if step < warmup:
        return max_lr * (step + 1) / warmup
    progress = (step - warmup) / max(1, max_steps - warmup)
    return max_lr * (min_ratio + (1 - min_ratio) * 0.5 * (1 + math.cos(math.pi * min(progress, 1.0))))

steps = range(0, TOTAL_STEPS, max(1, TOTAL_STEPS // 1000))
plt.figure(figsize=(6, 2.5))
plt.plot(steps, [lr_at(s, LR, WARMUP, TOTAL_STEPS) for s in steps])
plt.title("Learning-rate schedule over one full pass (same formula as run.py)"); plt.xlabel("step"); plt.ylabel("lr")
plt.grid(alpha=.3); plt.show()

In [ ]:
import subprocess, signal

LOG_PATH, PID_PATH = WORK_DIR / "run.log", WORK_DIR / "run.pid"

def running():
    try:
        os.kill(int(PID_PATH.read_text()), 0)
        return True
    except Exception:
        return False

if running():
    print("run.py is already running (pid", PID_PATH.read_text() + ") → just watch it with the monitor cell")
else:
    env = dict(os.environ, HF_TOKEN=HF_TOKEN or "", PYTHONUNBUFFERED="1")
    proc = subprocess.Popen([sys.executable, "-u", "run.py"], cwd=WORK_DIR, env=env,
                            stdout=open(LOG_PATH, "a"), stderr=subprocess.STDOUT,
                            start_new_session=True)              # own session: survives kernel restarts / page reloads
    PID_PATH.write_text(str(proc.pid))
    print("🚀 started run.py in the background, pid", proc.pid)

In [ ]:
# 👀 Monitor: re-run this cell any time (e.g. after a page reload). Stop it with ■; that does NOT stop training.
from IPython.display import clear_output
while True:
    clear_output(wait=True)
    st = json.loads((WORK_DIR / "status.json").read_text()) if (WORK_DIR / "status.json").exists() else {}
    print(f"run.py {'RUNNING' if running() else 'NOT running'} | phase: {st.get('phase', '?')} | "
          f"{st.get('pct', 0):.2%} | {time.strftime('%H:%M:%S')}\n")
    if LOG_PATH.exists():
        print("".join(LOG_PATH.read_text(errors="ignore").splitlines(keepends=True)[-25:]))
    if st.get("phase") == "done" or not running():
        break
    time.sleep(60)

In [ ]:
# ⏹ Stop the background job (only if you really want to; progress up to the last checkpoint is kept)
STOP = False
if STOP and running():
    os.killpg(int(PID_PATH.read_text()), signal.SIGTERM)
    print("stopped")

### ✅ Coverage check: did we really use the whole tar?

This reads `progress.json` (written by `run.py`): how many `.txt` files have been tokenized and **how far into the tar file** the stream has read. The job is only complete when it has read **to the last byte of the file**. Run it any time.

In [ ]:
import importlib, llm_work.llm_lib as L; importlib.reload(L)
if not (WORK_DIR / "progress.json").exists():
    hf_fetch("progress.json", WORK_DIR)
P = json.loads((WORK_DIR / "progress.json").read_text())
print(f"files tokenized : {P['files'] if P['files'] is not None else '(reused an existing tokens.bin)'}")
print(f"tar bytes read  : {P['offset']:>12,} / {L.DATASET_BYTES:,} ({P['offset']/L.DATASET_BYTES:.2%})")
print(f"tokens          : {P['tokens']:>12,} in {len(P['shards'])} shards")
print(f"finished        : {P['done']}")
if P["done"] and P["offset"] >= L.DATASET_BYTES - L.TAR_TRAILER:
    print("\n✅ Read to the last byte of the tar: EVERY .txt file has been tokenized.")
else:
    print("\n⏳ not complete yet: run.py is still working (or re-launch it to continue).")

### Load the trained weights into the notebook

When the monitor says **TRAINING DONE** (or at any time, to look at a checkpoint), load `gpt_from_scratch.pt` into the notebook's `model` and plot the loss curves. The parameter names are identical, so the weights drop straight in.

In [ ]:
MODEL_PATH = WORK_DIR / "gpt_from_scratch.pt"
if not MODEL_PATH.exists() and USE_HF:
    hf_fetch("gpt_from_scratch.pt", WORK_DIR); hf_fetch("train_state.pt", WORK_DIR)
if not MODEL_PATH.exists():
    st = json.loads((WORK_DIR / "status.json").read_text()) if (WORK_DIR / "status.json").exists() else {}
    tail = "".join(LOG_PATH.read_text(errors="ignore").splitlines(keepends=True)[-15:]) if LOG_PATH.exists() else "(no run.log yet)"
    if running():
        msg = (f"⏳ No checkpoint yet: run.py is still working (phase: {st.get('phase', '?')}, {st.get('pct', 0):.1%}).\n"
               "   The first checkpoint is written at the first progress report of training. Re-run this cell later.")
    else:
        msg = "❌ No checkpoint, and run.py is NOT running: it stopped. The end of run.log says why:"
    raise SystemExit(f"{msg}\n\n--- last lines of run.log ---\n{tail}")
ckpt = torch.load(MODEL_PATH, map_location=device, weights_only=False)
model = GPT(ckpt["cfg"]).to(device)
model.load_state_dict(ckpt["model"])
print(f"loaded weights after {ckpt['step']:,} batches ({ckpt['tokens_seen']:,} tokens)")

history = torch.load(WORK_DIR / "train_state.pt", map_location="cpu", weights_only=False)["history"]
plt.figure(figsize=(8, 4))
plt.plot(history["step"], history["train"], label="train (running avg)")
plt.plot(history["step"], history["val"], label="validation")
plt.xlabel("batch"); plt.ylabel("cross-entropy loss"); plt.title("Training progress"); plt.legend(); plt.grid(alpha=.3)
plt.show()

**Reading the curve.** Both losses should fall together. Because the model never sees the same book twice, it can't memorize, so the train and validation losses stay close. The gap is **overfitting**, and it only opens up if you train on the same text over and over.

Perplexity `= e^loss` is roughly "how many tokens the model is choosing between". Lower is better.

## 9. Generate text ✍️

Generation is a loop: feed the prompt, take the probabilities for the **last** position, **sample** a token, append it, and repeat.

```
 "Once upon a"  ──model──▶ P(next) = { " time": 0.62, " day": 0.08, " hill": 0.03, ... } ──sample──▶ " time"
 "Once upon a time" ──model──▶ ...
```

Two knobs:

* **temperature**: `<1` is safer and more repetitive, `>1` is more creative and more chaotic,
* **top-k**: sample only from the `k` most likely tokens.

In [ ]:
torch.manual_seed(42)
for temp in [0.5, 0.8, 1.2]:
    print(f"━━━ temperature {temp} ━━━")
    print(complete("Once upon a time", max_new_tokens=100, temperature=temp, top_k=50))
    print()

In [ ]:
# ✏️ Try your own prompt!
prompt = "She opened the door and"
print(complete(prompt, max_new_tokens=150, temperature=0.8, top_k=40))

### Peek inside: what does the trained model attend to?

We run one sentence through the first block and plot each head's attention weights. After training, heads usually specialize: some look at the previous token, some at the start of the sentence, some at matching words.

In [ ]:
sentence = "The old man looked at the sea and smiled."
ids = torch.tensor([tokenizer.encode(sentence)], device=device)
labels = [tokenizer.decode([i]) for i in ids[0].tolist()]

model.eval()
with torch.no_grad():
    x = model.tok_emb(ids) + model.pos_emb(torch.arange(ids.shape[1], device=device))
    _, att = model.blocks[0].attn(model.blocks[0].ln1(x), return_attn=True)   # (1, heads, T, T)

n_show = min(4, cfg["n_heads"])
fig, axes = plt.subplots(1, n_show, figsize=(4.2 * n_show, 4))
for h_i, ax in enumerate(axes):
    ax.imshow(att[0, h_i].float().cpu(), cmap="viridis")
    ax.set_xticks(range(len(labels)), labels, rotation=90, fontsize=8)
    ax.set_yticks(range(len(labels)), labels, fontsize=8)
    ax.set_title(f"layer 1 · head {h_i+1}")
plt.tight_layout(); plt.show()

### Which tokens ended up with similar embeddings?

Similar words should now have similar embedding vectors. Here are the nearest neighbours by cosine similarity:

In [ ]:
E = F.normalize(model.tok_emb.weight.detach().float(), dim=-1)
for word in [" king", " love", " night", " said", " house"]:
    ids_w = tokenizer.encode(word)
    if len(ids_w) != 1:
        print(f"{word!r} is split into {len(ids_w)} tokens, skipping"); continue
    sims = E @ E[ids_w[0]]
    best = sims.topk(8).indices.tolist()[1:]
    print(f"{word!r:>10} → {[tokenizer.decode([b]) for b in best]}")

## 10. Save and download the model 💾

Files in `llm_work/`:

| File | What |
|---|---|
| `tokens_00000.bin` … | the whole tar as token ids, 1 GB shards (also on Hugging Face) |
| `progress.json` | how far the tar has been tokenized (byte offset, files, tokens) |
| `tokenizer.json` | the BPE merges + special tokens |
| `gpt_from_scratch.pt` | model weights + config (small) |
| `train_state.pt` | weights + optimizer + step + loss history (for resuming) |
| `run.log` | everything the background job printed |

**Reasoning_Model_From_Scratch.ipynb** builds on this model. It downloads `gpt_from_scratch.pt` + `tokenizer.json` from your Hugging Face repo automatically. As a backup, this cell also downloads them to your computer.

In [ ]:
ckpt = torch.load(MODEL_PATH, map_location=device, weights_only=False)
model2 = GPT(ckpt["cfg"]).to(device)
model2.load_state_dict(ckpt["model"])
tok2 = BPETokenizer.load(TOK_PATH)
assert tok2.encode("Hello world") == tokenizer.encode("Hello world")
print(f"reloaded model ({MODEL_PATH.stat().st_size/1e6:.0f} MB) + tokenizer ✓ | trained on {ckpt['tokens_seen']:,} tokens")

if IN_COLAB:
    from google.colab import files
    files.download(str(MODEL_PATH))
    files.download(str(TOK_PATH))

## 🎉 Recap and next steps

You built every part of a GPT by hand:

```
 books ─▶ BPE tokenizer ─▶ sliding windows ─▶ embeddings ─▶ [ attention + MLP ] × N ─▶ logits ─▶ next token
```

**Ideas to go further**

1. **Scale up**: more books, `d_model=768, n_layers=12, n_heads=12` (GPT-2 small shape), and more steps.
2. **Better tokenizer**: a bigger `VOCAB_SIZE` (8k–32k) once you have lots of text.
3. **Modern tricks**: RoPE positional encoding, RMSNorm, SwiGLU MLPs, and a KV-cache for fast generation (the Llama recipe).
4. **Fine-tuning**: continue training on a smaller, specific text (one author, or dialogue) at a lower learning rate.
5. **Instruction tuning**: train on `### Instruction / ### Response` pairs to make a small chat model.

**References:** Vaswani et al., *Attention Is All You Need* (2017) · Radford et al., *GPT-2* (2019) · Sebastian Raschka, *Build a Large Language Model (From Scratch)* · Andrej Karpathy, *nanoGPT* and *minbpe*.